In [55]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier

In [56]:
df = pd.read_csv('../data/Telco-Customer-Churn-Engineered.csv')
df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,is_high_risk_billing,Total_Services,gender_Male,Partner_Yes,Dependents_Yes,...,DeviceProtection_Yes,TechSupport_Yes,StreamingTV_Yes,StreamingMovies_Yes,PaperlessBilling_Yes,MultipleLines,InternetService,Contract,PaymentMethod,tenure_year
0,0,1,29.85,29.85,0,1,1,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,2.0,0.0
1,0,34,56.95,1889.50,0,0,3,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3.0,2.0
2,0,2,53.85,108.15,1,0,3,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,3.0,0.0
3,0,45,42.30,1840.75,0,0,3,1.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,3.0
4,0,2,70.70,151.65,1,1,1,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2.0,0.0


In [57]:
X = df.drop('Churn', axis=1)
y = df['Churn']

In [58]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify= y)

In [59]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [60]:
random_forest = RandomForestClassifier(n_estimators=100)
random_forest.fit(X_train, y_train)
y_pred = random_forest.predict(X_test)
print(classification_report(y_test, y_pred))
print("--------------------------------------------------")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.91      0.86      1033
           1       0.64      0.47      0.54       374

    accuracy                           0.79      1407
   macro avg       0.73      0.69      0.70      1407
weighted avg       0.78      0.79      0.78      1407

--------------------------------------------------
[[936  97]
 [200 174]]


In [61]:
for feature, importance in zip(X.columns, random_forest.feature_importances_):
    print(f"{feature}: {importance}")

SeniorCitizen: 0.021578217539793075
tenure: 0.13592597146779858
MonthlyCharges: 0.17344384703101112
TotalCharges: 0.1668241947764063
is_high_risk_billing: 0.051967373520312166
Total_Services: 0.03968464708154628
gender_Male: 0.027569524011406252
Partner_Yes: 0.022754844306102213
Dependents_Yes: 0.018120813653008366
PhoneService_Yes: 0.005201062479957212
OnlineSecurity_Yes: 0.01975748467986474
OnlineBackup_Yes: 0.018490645018993414
DeviceProtection_Yes: 0.01746992388537985
TechSupport_Yes: 0.017973955340983398
StreamingTV_Yes: 0.01534576185352099
StreamingMovies_Yes: 0.015857515290074037
PaperlessBilling_Yes: 0.02516817422789651
MultipleLines: 0.022005417896683425
InternetService: 0.03415799948656895
Contract: 0.06788391713547753
PaymentMethod: 0.03523958567671697
tenure_year: 0.047579123640498634


In [62]:
random_forest = RandomForestClassifier(n_estimators=100, class_weight='balanced')
random_forest.fit(X_train, y_train)
y_pred = random_forest.predict(X_test)
print(classification_report(y_test, y_pred))
print("--------------------------------------------------")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.92      0.87      1033
           1       0.66      0.45      0.53       374

    accuracy                           0.79      1407
   macro avg       0.74      0.68      0.70      1407
weighted avg       0.78      0.79      0.78      1407

--------------------------------------------------
[[948  85]
 [207 167]]


In [63]:
import xgboost as xgb
xgb = XGBClassifier(n_estimators=100)
xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.89      0.85      1033
           1       0.59      0.46      0.52       374

    accuracy                           0.77      1407
   macro avg       0.71      0.67      0.69      1407
weighted avg       0.76      0.77      0.76      1407

[[915 118]
 [201 173]]


In [64]:
from sklearn.model_selection import RandomizedSearchCV

params = {
    'learning_rate': [0.1, 0.01, 0.001, 0.0001],
    'max_depth': [3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    'n_estimators': [100, 200, 300, 400, 500],
    'scale_pos_weight' : [2, 3, 4, 5, 6]
}

xgb = XGBClassifier(random_state=42)
randomized_search = RandomizedSearchCV(estimator=xgb, param_distributions=params, n_iter=100, scoring='f1', cv=5)
randomized_search.fit(X_train, y_train)
y_pred = randomized_search.best_estimator_.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("Best parameters: ", randomized_search.best_params_)
print("Best score: ", randomized_search.best_score_)

              precision    recall  f1-score   support

           0       0.88      0.79      0.83      1033
           1       0.55      0.70      0.62       374

    accuracy                           0.77      1407
   macro avg       0.72      0.75      0.73      1407
weighted avg       0.79      0.77      0.78      1407

[[819 214]
 [112 262]]
Best parameters:  {'scale_pos_weight': 2, 'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.01}
Best score:  0.6399684318513741


In [65]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'learning_rate': [0.005, 0.01, 0.02],
    'max_depth': [3, 4, 5, 6],
    'n_estimators': [300, 400, 500],
    'scale_pos_weight' : [1, 2, 3, 4]
}
xgb = XGBClassifier(random_state=42)
grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, n_jobs=-1, scoring='f1', cv=5)
grid_search.fit(X_train, y_train)
y_pred = grid_search.best_estimator_.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print("Best parameters: ", grid_search.best_params_)
print("Best score: ", grid_search.best_score_)


              precision    recall  f1-score   support

           0       0.88      0.80      0.84      1033
           1       0.56      0.71      0.62       374

    accuracy                           0.77      1407
   macro avg       0.72      0.75      0.73      1407
weighted avg       0.80      0.77      0.78      1407

[[822 211]
 [109 265]]
Best parameters:  {'learning_rate': 0.01, 'max_depth': 4, 'n_estimators': 400, 'scale_pos_weight': 2}
Best score:  0.640858718474749


In [66]:
import joblib

best_model = grid_search.best_estimator_
joblib.dump(best_model, '../models/telco_churn_xgb_model.pkl')

['../models/telco_churn_xgb_model.pkl']

In [67]:
X_train.columns.tolist()

['SeniorCitizen',
 'tenure',
 'MonthlyCharges',
 'TotalCharges',
 'is_high_risk_billing',
 'Total_Services',
 'gender_Male',
 'Partner_Yes',
 'Dependents_Yes',
 'PhoneService_Yes',
 'OnlineSecurity_Yes',
 'OnlineBackup_Yes',
 'DeviceProtection_Yes',
 'TechSupport_Yes',
 'StreamingTV_Yes',
 'StreamingMovies_Yes',
 'PaperlessBilling_Yes',
 'MultipleLines',
 'InternetService',
 'Contract',
 'PaymentMethod',
 'tenure_year']